#  VBB Transit Data — Apache Spark Processing
**Project:** VBB Transit Data Engineering and Analytics Platform  
**Engine:** Apache Spark 4.1.2 (PySpark)  
**Source:** PostgreSQL — vbb_db  
**Team:** Pradeep | Abhishek | Mahesh | Nisarga  
**Professor:** Dr. Farhan Khan


# What We Compute Here
- Route coverage and transport mode analysis
- Stop density and geographic clustering
- Transfer hub rankings at scale
- Service reliability patterns
- Accessibility statistics across the full network

---
## 0. Setup — Start Spark Session & Connect to PostgreSQL

In [26]:
import os
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Path to the JDBC driver we downloaded
JDBC_JAR = os.path.expanduser('~/Desktop/postgresql-42.7.3.jar')

# Start Spark Session
spark = SparkSession.builder \
    .appName('VBB Transit Analytics') \
    .config('spark.jars', JDBC_JAR) \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

print('⚡ Spark Session started successfully!')
print(f'   Spark version : {spark.version}')
print(f'   App name      : {spark.sparkContext.appName}')
print(f'   JDBC driver   : {JDBC_JAR}')

⚡ Spark Session started successfully!
   Spark version : 4.1.2
   App name      : VBB Transit Analytics
   JDBC driver   : /Users/abhishekkarthikakunuru/Desktop/postgresql-42.7.3.jar


In [27]:
import os

# PostgreSQL connection settings
DB_URL   = 'jdbc:postgresql://localhost:5432/vbb_db'
DB_PROPS = {
    'driver':   'org.postgresql.Driver',
    'user':     os.environ.get('USER', 'abhishekkarthikakunuru'),
    'password': ''
}

def read_table(table_name):
    df = spark.read.jdbc(
        url=DB_URL,
        table=table_name,
        properties=DB_PROPS
    )
    return df

OUTPUT_DIR = os.path.expanduser(
    '~/Desktop/Data Engineering /Spark_Output/'
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ PostgreSQL connection configured')
print(f'   User      : {DB_PROPS["user"]}')
print(f'   Database  : vbb_db')
print(f'   Output    : {OUTPUT_DIR}')

✅ PostgreSQL connection configured
   User      : abhishekkarthikakunuru
   Database  : vbb_db
   Output    : /Users/abhishekkarthikakunuru/Desktop/Data Engineering /Spark_Output/


---
## 1. Load All 8 Tables from PostgreSQL into Spark

In [28]:
# Load all tables — this is where Spark shines
# Each table becomes a distributed DataFrame
print('Loading tables from PostgreSQL into Spark...\n')

tables = ['agency','routes','stops','calendar',
          'calendar_dates','transfers','pathways','levels']

dfs = {}
for name in tables:
    dfs[name] = read_table(name)
    print(f'  ✅  {name:<20} → {dfs[name].count():>7,} rows  |  {len(dfs[name].columns)} columns')

print(f'\n⚡ All tables loaded into Spark!')

Loading tables from PostgreSQL into Spark...

  ✅  agency               →      35 rows  |  6 columns
  ✅  routes               →   1,322 rows  |  9 columns
  ✅  stops                →  41,840 rows  |  14 columns
  ✅  calendar             →   3,327 rows  |  15 columns
  ✅  calendar_dates       →  75,729 rows  |  7 columns
  ✅  transfers            →  52,682 rows  |  11 columns
  ✅  pathways             → 125,940 rows  |  9 columns
  ✅  levels               →     203 rows  |  4 columns

⚡ All tables loaded into Spark!


---
## 2. Transport Mode Analysis at Scale

In [29]:
# Routes joined with Agency — distributed join across two tables
routes_with_agency = dfs['routes'].join(
    dfs['agency'].select('agency_id', 'agency_name'),
    on='agency_id',
    how='left'
)

# Transport mode breakdown
mode_breakdown = routes_with_agency \
    .groupBy('transport_mode') \
    .agg(F.count('route_id').alias('route_count')) \
    .orderBy(F.desc('route_count'))

print('🚌 Transport Mode Breakdown (computed by Spark)\n')
mode_breakdown.show(truncate=False)

# Save as Parquet
mode_breakdown.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'mode_breakdown.parquet'
)
print('✅ Saved as Parquet → mode_breakdown.parquet')

🚌 Transport Mode Breakdown (computed by Spark)

+---------------+-----------+
|transport_mode |route_count|
+---------------+-----------+
|Bus            |1138       |
|Rail (Regional)|69         |
|Tram           |50         |
|S-Bahn         |48         |
|U-Bahn         |9          |
|Water Transport|8          |
+---------------+-----------+

✅ Saved as Parquet → mode_breakdown.parquet


In [30]:
# Top 15 agencies by number of routes
agency_routes = routes_with_agency \
    .groupBy('agency_name') \
    .agg(F.count('route_id').alias('route_count')) \
    .orderBy(F.desc('route_count')) \
    .limit(15)

print('🏢 Top 15 Agencies by Route Count\n')
agency_routes.show(truncate=False)

agency_routes.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'agency_routes.parquet'
)
print('✅ Saved → agency_routes.parquet')

🏢 Top 15 Agencies by Route Count

+------------------------------------------------------------+-----------+
|agency_name                                                 |route_count|
+------------------------------------------------------------+-----------+
|Berliner Verkehrsbetriebe                                   |265        |
|prignitzbus                                                 |85         |
|Cottbusverkehr GmbH                                         |76         |
|regiobus Potsdam Mittelmark GmbH                            |74         |
|Oberhavel Verkehrsgesellschaft mbH                          |68         |
|Uckermärkische Verkehrsgesellschaft mbH                     |67         |
|Barnimer Busgesellschaft mbH                                |64         |
|S-Bahn Berlin GmbH                                          |57         |
|Havelbus Verkehrsgesellschaft mbH                           |56         |
|Verkehrsgesellschaft Teltow-Fläming mbH                     |55  

---
## 3. Stop Analysis — Density, Types and Accessibility

In [31]:
# Stop type distribution
stop_types = dfs['stops'] \
    .groupBy('location_type_name') \
    .agg(F.count('stop_id').alias('count')) \
    .orderBy(F.desc('count'))

print('📍 Stop Types Across the Network\n')
stop_types.show(truncate=False)

📍 Stop Types Across the Network

+------------------+-----+
|location_type_name|count|
+------------------+-----+
|Stop / Platform   |27767|
|Generic Node      |6774 |
|Entrance / Exit   |6422 |
|Station           |877  |
+------------------+-----+



In [32]:
# Accessibility statistics
total      = dfs['stops'].count()
accessible = dfs['stops'].filter(F.col('is_accessible') == True).count()
pct        = round(accessible / total * 100, 1)

print('♿ Accessibility Analysis\n')
print(f'  Total stops        : {total:,}')
print(f'  Accessible stops   : {accessible:,}  ({pct}%)')
print(f'  Not accessible     : {total - accessible:,}  ({round(100-pct,1)}%)')
print(f'\n  ⚠️  {round(100-pct,1)}% of VBB stops are NOT wheelchair accessible')
print(f'  This is a key finding for Business Question 2!')

♿ Accessibility Analysis

  Total stops        : 41,840
  Accessible stops   : 3,235  (7.7%)
  Not accessible     : 38,605  (92.3%)

  ⚠️  92.3% of VBB stops are NOT wheelchair accessible
  This is a key finding for Business Question 2!


In [33]:
# Geographic clustering — divide Berlin into grid zones
# Round coordinates to 1 decimal place to create zones
stop_density = dfs['stops'] \
    .withColumn('lat_zone', F.round(F.col('stop_lat'), 1)) \
    .withColumn('lon_zone', F.round(F.col('stop_lon'), 1)) \
    .groupBy('lat_zone', 'lon_zone') \
    .agg(
        F.count('stop_id').alias('stop_count'),
        F.sum(F.when(F.col('is_accessible'), 1).otherwise(0)).alias('accessible_count')
    ) \
    .orderBy(F.desc('stop_count'))

print('🗺️  Top 15 Stop Density Zones\n')
stop_density.show(15, truncate=False)

stop_density.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'stop_density.parquet'
)
print('✅ Saved → stop_density.parquet')

🗺️  Top 15 Stop Density Zones

+--------+--------+----------+----------------+
|lat_zone|lon_zone|stop_count|accessible_count|
+--------+--------+----------+----------------+
|52.5    |13.4    |5168      |4               |
|52.5    |13.3    |3066      |2               |
|52.5    |13.5    |2428      |4               |
|52.6    |13.4    |1511      |0               |
|52.5    |13.6    |1351      |19              |
|52.4    |13.5    |952       |0               |
|52.6    |13.3    |840       |1               |
|52.4    |13.4    |814       |20              |
|52.6    |13.5    |806       |8               |
|52.5    |13.2    |734       |3               |
|52.4    |13.3    |677       |26              |
|52.4    |13.1    |675       |169             |
|52.4    |13.6    |556       |11              |
|52.4    |13.2    |457       |0               |
|52.4    |13.0    |335       |55              |
+--------+--------+----------+----------------+
only showing top 15 rows
✅ Saved → stop_density.parquet


---
## 4. Transfer Hub Analysis — Busiest Interchange Stations

In [34]:
# Step 1: Count transfers per stop
transfer_counts = dfs['transfers'] \
    .groupBy('from_stop_id') \
    .agg(F.count('*').alias('transfer_count'))

# Step 2: Prepare stops with stop_id as string
stops_clean = dfs['stops'].select(
    F.col('stop_id').cast('string').alias('stop_id_str'),
    'stop_name',
    'location_type_name',
    'is_accessible',
    'stop_lat',
    'stop_lon'
)

# Step 3: Join the two DataFrames cleanly
transfer_hubs = transfer_counts.join(
    stops_clean,
    transfer_counts['from_stop_id'] == stops_clean['stop_id_str'],
    how='left'
) \
.orderBy(F.desc('transfer_count')) \
.limit(20)

print('🔀 Top 20 Transfer Hubs (computed by Spark)\n')
transfer_hubs.select(
    'stop_name',
    'transfer_count',
    'location_type_name',
    'is_accessible'
).show(truncate=False)

# Save as Parquet
transfer_hubs.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'transfer_hubs_ranked.parquet'
)
print('✅ Saved → transfer_hubs_ranked.parquet')

🔀 Top 20 Transfer Hubs (computed by Spark)

+-------------------------------------+--------------+------------------+-------------+
|stop_name                            |transfer_count|location_type_name|is_accessible|
+-------------------------------------+--------------+------------------+-------------+
|Zehlendorf Eiche (Berlin)            |205           |Stop / Platform   |false        |
|Fürstenwalde, Bahnhof                |189           |Stop / Platform   |false        |
|S+U Rathaus Spandau (Berlin)         |182           |Stop / Platform   |false        |
|S+U Rathaus Spandau (Berlin)         |156           |Stop / Platform   |false        |
|S+U Rathaus Spandau (Berlin)         |156           |Stop / Platform   |false        |
|S+U Rathaus Spandau (Berlin)         |154           |Stop / Platform   |false        |
|Zehlendorf Eiche (Berlin)            |126           |Stop / Platform   |false        |
|S+U Rathaus Spandau (Berlin)         |105           |Stop / Platform   |fal

In [35]:
# Simpler transfer hub ranking using Window functions
# Window functions are a key Spark feature — like SQL RANK()
transfer_counts = dfs['transfers'] \
    .groupBy('from_stop_id') \
    .agg(F.count('*').alias('outbound_transfers'))

# Add rank using Window
window = Window.orderBy(F.desc('outbound_transfers'))
ranked = transfer_counts \
    .withColumn('rank', F.rank().over(window)) \
    .filter(F.col('rank') <= 20)

print('🏆 Top 20 Stops by Outbound Transfers (with Rank)\n')
ranked.show(truncate=False)

ranked.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'transfer_hubs_ranked.parquet'
)
print('✅ Saved → transfer_hubs_ranked.parquet')

🏆 Top 20 Stops by Outbound Transfers (with Rank)

+----------------------+------------------+----+
|from_stop_id          |outbound_transfers|rank|
+----------------------+------------------+----+
|de:11000:900049241::3 |205               |1   |
|de:12067:900310001::11|189               |2   |
|de:11000:900029371::2 |182               |3   |
|de:11000:900029302::5 |156               |4   |
|de:11000:900029302::4 |156               |4   |
|de:11000:900029302::6 |154               |6   |
|de:11000:900049271::1 |126               |7   |
|de:11000:900029302::13|105               |8   |
|de:12069:900220019::9 |99                |9   |
|de:12052:900470268::3 |96                |10  |
|de:12069:900220019::1 |93                |11  |
|de:11000:900029302::10|87                |12  |
|de:11000:900029302::7 |86                |13  |
|de:12052:900470268::4 |81                |14  |
|de:11000:900056101::7 |73                |15  |
|de:12067:900311129::6 |73                |15  |
|de:12067:900310013

---
## 5. Service Reliability — Calendar Analysis at Scale

In [36]:
# Service pattern distribution
patterns = dfs['calendar'] \
    .groupBy('service_pattern') \
    .agg(F.count('service_id').alias('service_count')) \
    .orderBy(F.desc('service_count'))

print('📅 Service Pattern Distribution\n')
patterns.show(truncate=False)

📅 Service Pattern Distribution

+---------------+-------------+
|service_pattern|service_count|
+---------------+-------------+
|No Service     |2065         |
|Full Week      |585          |
|Weekdays Only  |377          |
|Weekends Only  |300          |
+---------------+-------------+



In [37]:
# Which day of week has most service exceptions?
exceptions_by_day = dfs['calendar_dates'] \
    .groupBy('day_of_week', 'exception_label') \
    .agg(F.count('*').alias('exception_count')) \
    .orderBy(F.desc('exception_count'))

print('📆 Service Exceptions by Day of Week\n')
exceptions_by_day.show(14, truncate=False)

exceptions_by_day.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'exceptions_by_day.parquet'
)
print('✅ Saved → exceptions_by_day.parquet')

📆 Service Exceptions by Day of Week

+-----------+---------------+---------------+
|day_of_week|exception_label|exception_count|
+-----------+---------------+---------------+
|Thursday   |Service Added  |7655           |
|Friday     |Service Added  |7561           |
|Monday     |Service Added  |6970           |
|Tuesday    |Service Added  |6902           |
|Wednesday  |Service Added  |6646           |
|Saturday   |Service Added  |6292           |
|Sunday     |Service Added  |5576           |
|Monday     |Service Removed|4369           |
|Friday     |Service Removed|4369           |
|Thursday   |Service Removed|4254           |
|Tuesday    |Service Removed|4129           |
|Wednesday  |Service Removed|4005           |
|Saturday   |Service Removed|3655           |
|Sunday     |Service Removed|3346           |
+-----------+---------------+---------------+

✅ Saved → exceptions_by_day.parquet


In [38]:
# Services added vs removed by month
exceptions_by_month = dfs['calendar_dates'] \
    .groupBy('month', 'year', 'exception_label') \
    .agg(F.count('*').alias('count')) \
    .orderBy('year', F.desc('count'))

print('📆 Service Exceptions by Month\n')
exceptions_by_month.show(20, truncate=False)

exceptions_by_month.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'exceptions_by_month.parquet'
)
print('✅ Saved → exceptions_by_month.parquet')

📆 Service Exceptions by Month

+---------+----+---------------+-----+
|month    |year|exception_label|count|
+---------+----+---------------+-----+
|May      |2026|Service Added  |18426|
|June     |2026|Service Added  |13933|
|May      |2026|Service Removed|9223 |
|July     |2026|Service Added  |5172 |
|October  |2026|Service Removed|4883 |
|June     |2026|Service Removed|3523 |
|November |2026|Service Removed|3213 |
|October  |2026|Service Added  |2344 |
|November |2026|Service Added  |2306 |
|August   |2026|Service Added  |2237 |
|July     |2026|Service Removed|2135 |
|August   |2026|Service Removed|1911 |
|September|2026|Service Added  |1720 |
|December |2026|Service Removed|1520 |
|September|2026|Service Removed|1430 |
|December |2026|Service Added  |835  |
|April    |2026|Service Added  |629  |
|April    |2026|Service Removed|289  |
+---------+----+---------------+-----+

✅ Saved → exceptions_by_month.parquet


---
## 6. Station Infrastructure — Pathway Analysis

In [39]:
# Pathway types across the network
pathway_types = dfs['pathways'] \
    .groupBy('pathway_mode_name', 'direction') \
    .agg(
        F.count('*').alias('count'),
        F.round(F.avg('traversal_time'), 1).alias('avg_traversal_secs'),
        F.round(F.avg('length'), 1).alias('avg_length_metres')
    ) \
    .orderBy(F.desc('count'))

print('🚶 Pathway Infrastructure Analysis\n')
pathway_types.show(truncate=False)

pathway_types.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'pathway_types.parquet'
)
print('✅ Saved → pathway_types.parquet')

🚶 Pathway Infrastructure Analysis

+-----------------+---------+------+------------------+-----------------+
|pathway_mode_name|direction|count |avg_traversal_secs|avg_length_metres|
+-----------------+---------+------+------------------+-----------------+
|Walkway          |One-Way  |119231|81.7              |113.5            |
|Stairs           |One-Way  |4410  |20.3              |1.4              |
|Elevator         |One-Way  |1544  |81.9              |5.9              |
|Escalator        |One-Way  |755   |30.4              |1.2              |
+-----------------+---------+------+------------------+-----------------+

✅ Saved → pathway_types.parquet


---
## 7. Big Join — All Tables Combined

In [40]:
# Master analytical view — join routes + agency + stops via transfers
# This is a multi-table join that Spark handles in parallel

# Step 1: Routes enriched with agency
routes_enriched = dfs['routes'].join(
    dfs['agency'].select('agency_id', 'agency_name'),
    on='agency_id', how='left'
)

# Step 2: Transfer counts per stop
transfer_counts = dfs['transfers'] \
    .groupBy('from_stop_id') \
    .agg(F.count('*').alias('transfer_count'))

# Step 3: Stops enriched with transfer counts
stops_enriched = dfs['stops'].join(
    transfer_counts,
    dfs['stops']['stop_id'].cast('string') == transfer_counts['from_stop_id'],
    how='left'
).fillna({'transfer_count': 0})

print('✅ Master analytical views created:')
print(f'   routes_enriched  → {routes_enriched.count():,} rows')
print(f'   stops_enriched   → {stops_enriched.count():,} rows')

# Save both as Parquet
routes_enriched.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'routes_enriched.parquet'
)
stops_enriched.write.mode('overwrite').parquet(
    OUTPUT_DIR + 'stops_enriched.parquet'
)
print('\n✅ Both saved as Parquet files!')

✅ Master analytical views created:
   routes_enriched  → 1,322 rows
   stops_enriched   → 41,840 rows

✅ Both saved as Parquet files!


---
## 8. Spark SQL — Query Using SQL Syntax

In [41]:
# Spark SQL lets you write regular SQL on top of DataFrames
# Register DataFrames as temporary SQL tables
dfs['stops'].createOrReplaceTempView('stops')
dfs['routes'].createOrReplaceTempView('routes')
dfs['transfers'].createOrReplaceTempView('transfers')
dfs['agency'].createOrReplaceTempView('agency')
dfs['calendar'].createOrReplaceTempView('calendar')

print('✅ All tables registered for Spark SQL')

✅ All tables registered for Spark SQL


In [42]:
# SQL Query 1 — Most common transport modes
result1 = spark.sql("""
    SELECT
        r.transport_mode,
        a.agency_name,
        COUNT(r.route_id) as route_count
    FROM routes r
    JOIN agency a ON r.agency_id = a.agency_id
    GROUP BY r.transport_mode, a.agency_name
    ORDER BY route_count DESC
    LIMIT 10
""")

print('🚌 SQL Query 1 — Top Routes by Mode and Agency\n')
result1.show(truncate=False)

🚌 SQL Query 1 — Top Routes by Mode and Agency

+--------------+---------------------------------------+-----------+
|transport_mode|agency_name                            |route_count|
+--------------+---------------------------------------+-----------+
|Bus           |Berliner Verkehrsbetriebe              |221        |
|Bus           |prignitzbus                            |85         |
|Bus           |regiobus Potsdam Mittelmark GmbH       |74         |
|Bus           |Cottbusverkehr GmbH                    |72         |
|Bus           |Oberhavel Verkehrsgesellschaft mbH     |68         |
|Bus           |Uckermärkische Verkehrsgesellschaft mbH|67         |
|Bus           |Barnimer Busgesellschaft mbH           |64         |
|Bus           |Havelbus Verkehrsgesellschaft mbH      |56         |
|Bus           |Verkehrsgesellschaft Teltow-Fläming mbH|55         |
|Bus           |Busverkehr Oder-Spree GmbH             |54         |
+--------------+---------------------------------------+

In [43]:
# SQL Query 2 — Accessibility by stop type
result2 = spark.sql("""
    SELECT
        location_type_name,
        COUNT(*) as total_stops,
        SUM(CASE WHEN is_accessible = true THEN 1 ELSE 0 END) as accessible,
        ROUND(
            SUM(CASE WHEN is_accessible = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            1
        ) as accessible_pct
    FROM stops
    GROUP BY location_type_name
    ORDER BY total_stops DESC
""")

print('♿ SQL Query 2 — Accessibility Rate by Stop Type\n')
result2.show(truncate=False)

♿ SQL Query 2 — Accessibility Rate by Stop Type

+------------------+-----------+----------+--------------+
|location_type_name|total_stops|accessible|accessible_pct|
+------------------+-----------+----------+--------------+
|Stop / Platform   |27767      |1084      |3.9           |
|Generic Node      |6774       |971       |14.3          |
|Entrance / Exit   |6422       |853       |13.3          |
|Station           |877        |327       |37.3          |
+------------------+-----------+----------+--------------+



In [44]:
# SQL Query 3 — Full week services vs weekday only
result3 = spark.sql("""
    SELECT
        service_pattern,
        COUNT(*) as service_count,
        ROUND(AVG(service_duration_days), 1) as avg_duration_days,
        ROUND(AVG(total_service_days_per_week), 1) as avg_days_per_week
    FROM calendar
    GROUP BY service_pattern
    ORDER BY service_count DESC
""")

print('📅 SQL Query 3 — Service Pattern Statistics\n')
result3.show(truncate=False)

📅 SQL Query 3 — Service Pattern Statistics

+---------------+-------------+-----------------+-----------------+
|service_pattern|service_count|avg_duration_days|avg_days_per_week|
+---------------+-------------+-----------------+-----------------+
|No Service     |2065         |226.0            |0.0              |
|Full Week      |585          |226.0            |5.7              |
|Weekdays Only  |377          |226.0            |4.1              |
|Weekends Only  |300          |226.0            |1.4              |
+---------------+-------------+-----------------+-----------------+



---
## 9. Summary — What Spark Produced

In [45]:
import os

# List all Parquet files created
parquet_files = [d for d in os.listdir(OUTPUT_DIR) if d.endswith('.parquet')]

print('='*55)
print('  ⚡ SPARK PROCESSING COMPLETE')
print('='*55)
print(f'  Tables processed   : 8')
print(f'  Total rows         : 301,078')
print(f'  Parquet files saved: {len(parquet_files)}')
print(f'  Output folder      : ~/Desktop/Data Engineering/Spark_Output/')
print()
print('  Files saved:')
for f in sorted(parquet_files):
    print(f'  ✅  {f}')
print()
print('  These Parquet files are ready for:')
print('  → Airflow DAG (next step)')
print('  → Grafana dashboard')
print('  → ML model training')
print('='*55)

# Stop the Spark session cleanly
spark.stop()
print('\n⚡ Spark session closed cleanly.')

  ⚡ SPARK PROCESSING COMPLETE
  Tables processed   : 8
  Total rows         : 301,078
  Parquet files saved: 9
  Output folder      : ~/Desktop/Data Engineering/Spark_Output/

  Files saved:
  ✅  agency_routes.parquet
  ✅  exceptions_by_day.parquet
  ✅  exceptions_by_month.parquet
  ✅  mode_breakdown.parquet
  ✅  pathway_types.parquet
  ✅  routes_enriched.parquet
  ✅  stop_density.parquet
  ✅  stops_enriched.parquet
  ✅  transfer_hubs_ranked.parquet

  These Parquet files are ready for:
  → Airflow DAG (next step)
  → Grafana dashboard
  → ML model training

⚡ Spark session closed cleanly.
